# GloGEM test: Aletsch & Morteratsch — output visualisation

This notebook reads and visualises GloGEM outputs from the six test runs.
Run the model first (from the GloGEM root in IDL):

```
cp test/config_aletsch_daily_calib.pro        config.pro  &&  .r glogem
cp test/config_aletsch_daily_hindcast.pro     config.pro  &&  .r glogem
cp test/config_aletsch_daily_projection.pro   config.pro  &&  .r glogem
cp test/config_aletsch_monthly_calib.pro      config.pro  &&  .r glogem
cp test/config_aletsch_monthly_hindcast.pro   config.pro  &&  .r glogem
cp test/config_aletsch_monthly_projection.pro config.pro  &&  .r glogem
```

Then open this notebook (in Jupyter or VS Code) and run all cells. Requires
only `numpy` and `matplotlib`.

## 1. Setup

Paths are detected automatically from the notebook's working directory —
no manual editing required (open/run this notebook from the `test/` folder).

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Locate the test directory and derive all paths
TEST_DIR = Path.cwd()
OUT_DIR = TEST_DIR / 'outputs'
CC = '_Aletsch_Morteratsch'   # catchment suffix in output filenames
PLOT_DIR = TEST_DIR / 'example_plots'
PLOT_DIR.mkdir(exist_ok=True)

required_files = [
    OUT_DIR / f'daily/CentralEurope/PAST/PAST_original/centraleurope_Annual_Balance_sfc_r1{CC}.dat',
    OUT_DIR / f'monthly/CentralEurope/PAST/PAST_original/centraleurope_Annual_Balance_sfc_r1{CC}.dat',
    OUT_DIR / f'daily/CentralEurope/files/files_original/MRI-ESM2-0/ssp126/centraleurope_Area_r1{CC}.dat',
    OUT_DIR / f'monthly/CentralEurope/files/files_original/MRI-ESM2-0/ssp126/centraleurope_Area_r1{CC}.dat',
]
missing = [f for f in required_files if not f.exists()]
if missing:
    for f in missing:
        print('MISSING:', f)
    print('Run the model steps first (see notebook header).')
else:
    print('Output files found - ready to visualise.')

## 2. Calibration results

Per-glacier calibrated parameters from both the daily and monthly runs.

In [ ]:
def read_calibration(path):
    with open(path) as f:
        header = f.readline()
        rows = [line.split() for line in f if line.strip()]
    return {
        'id':    np.array([r[0] for r in rows]),
        'ba':    np.array([float(r[1]) for r in rows]),
        'ela':   np.array([float(r[4]) for r in rows]),
        'aar':   np.array([float(r[5]) for r in rows]),
        'ddfs':  np.array([float(r[8]) for r in rows]),
        'ddfi':  np.array([float(r[9]) for r in rows]),
        'cprec': np.array([float(r[10]) for r in rows]),
        'toff':  np.array([float(r[11]) for r in rows]),
        'header': header,
    }

cal_d = read_calibration(OUT_DIR / f'daily/CentralEurope/calibration/calibrate_m1_cID9_centraleurope_final_era5{CC}.dat')
cal_m = read_calibration(OUT_DIR / f'monthly/CentralEurope/calibration/calibrate_m1_cID9_centraleurope_final_era5{CC}.dat')

idx_al_c  = int(np.where(cal_d['id'] == '02596')[0][0])
idx_mo_c  = int(np.where(cal_d['id'] == '02216')[0][0])
idx_al_cm = int(np.where(cal_m['id'] == '02596')[0][0])
idx_mo_cm = int(np.where(cal_m['id'] == '02216')[0][0])

print('Daily calibration:')
print(f"  Aletsch:     Ba={cal_d['ba'][idx_al_c]:6.3f}  ELA={cal_d['ela'][idx_al_c]:5.0f}  AAR={cal_d['aar'][idx_al_c]:5.1f}  DDFsnow={cal_d['ddfs'][idx_al_c]:.2f}  DDFice={cal_d['ddfi'][idx_al_c]:.2f}  c_prec={cal_d['cprec'][idx_al_c]:.2f}")
print(f"  Morteratsch: Ba={cal_d['ba'][idx_mo_c]:6.3f}  ELA={cal_d['ela'][idx_mo_c]:5.0f}  AAR={cal_d['aar'][idx_mo_c]:5.1f}  DDFsnow={cal_d['ddfs'][idx_mo_c]:.2f}  DDFice={cal_d['ddfi'][idx_mo_c]:.2f}  c_prec={cal_d['cprec'][idx_mo_c]:.2f}")

print()
print('Monthly calibration:')
print(f"  Aletsch:     Ba={cal_m['ba'][idx_al_cm]:6.3f}  DDFsnow={cal_m['ddfs'][idx_al_cm]:.2f}  DDFice={cal_m['ddfi'][idx_al_cm]:.2f}  c_prec={cal_m['cprec'][idx_al_cm]:.2f}")
print(f"  Morteratsch: Ba={cal_m['ba'][idx_mo_cm]:6.3f}  DDFsnow={cal_m['ddfs'][idx_mo_cm]:.2f}  DDFice={cal_m['ddfi'][idx_mo_cm]:.2f}  c_prec={cal_m['cprec'][idx_mo_cm]:.2f}")

## 3. Read hindcast time series

In [ ]:
def read_series(path):
    with open(path) as f:
        header = f.readline().split()
        years = np.array([float(y) for y in header[1:]])
        ids, data = [], []
        for line in f:
            if not line.strip():
                continue
            p = line.split()
            ids.append(p[0])
            data.append([float(v) for v in p[1:]])
    return np.array(ids), years, np.array(data)

# ---- Daily hindcast ----
gl_ids_d, yrs_d, mb_d = read_series(OUT_DIR / f'daily/CentralEurope/PAST/PAST_original/centraleurope_Annual_Balance_sfc_r1{CC}.dat')
idx_al = int(np.where(gl_ids_d == '02596')[0][0])
idx_mo = int(np.where(gl_ids_d == '02216')[0][0])

# ---- Monthly hindcast ----
gl_ids_m, yrs_m, mb_m = read_series(OUT_DIR / f'monthly/CentralEurope/PAST/PAST_original/centraleurope_Annual_Balance_sfc_r1{CC}.dat')
idx_al_m = int(np.where(gl_ids_m == '02596')[0][0])
idx_mo_m = int(np.where(gl_ids_m == '02216')[0][0])

# ---- ELA and AAR (daily) ----
_, _, ela_d = read_series(OUT_DIR / f'daily/CentralEurope/PAST/PAST_original/centraleurope_ELA_r1{CC}.dat')
_, _, aar_d = read_series(OUT_DIR / f'daily/CentralEurope/PAST/PAST_original/centraleurope_AAR_r1{CC}.dat')

print('Data loaded.')
print(f"  Daily   mean Ba - Aletsch: {mb_d[idx_al].mean():6.3f}   Morteratsch: {mb_d[idx_mo].mean():6.3f} m w.e./yr")
print(f"  Monthly mean Ba - Aletsch: {mb_m[idx_al_m].mean():6.3f}   Morteratsch: {mb_m[idx_mo_m].mean():6.3f} m w.e./yr")

## 4. Plots

In [ ]:
# ---- Plot 1: Daily annual mass balance ----
fig, ax = plt.subplots(figsize=(8.2, 4.2))
ax.plot(yrs_d, mb_d[idx_al], '-o', ms=3, color='#4682B4', label='Aletsch (daily)')
ax.plot(yrs_d, mb_d[idx_mo], '-o', ms=3, color='#D25A1E', label='Morteratsch (daily)')
ax.plot([1991, 2020], [-1.216, -1.216], ':', color='#4682B4', lw=1.5, label='Aletsch (Hugonnet et al. 2021)')
ax.plot([1991, 2020], [-1.022, -1.022], ':', color='#D25A1E', lw=1.5, label='Morteratsch (Hugonnet et al. 2021)')
ax.set_xlabel('Year'); ax.set_ylabel('Mass balance (m w.e.)'); ax.set_ylim(-5, 3)
ax.set_title('Daily model: Annual mass balance 1991-2020')
ax.legend(fontsize=9, loc='lower left')
fig.tight_layout()
fig.savefig(PLOT_DIR / 'plot1_annual_mb.png', dpi=150)
plt.show()

In [ ]:
# ---- Plot 2: ELA and AAR ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9.6, 4.0))
ax1.plot(yrs_d, ela_d[idx_al], 'k-o', ms=3)
ax1.set_title('Equilibrium Line Altitude'); ax1.set_xlabel('Year'); ax1.set_ylabel('ELA (m a.s.l.)')
ax2.plot(yrs_d, aar_d[idx_al], 'k--o', ms=3)
ax2.set_title('Accumulation Area Ratio'); ax2.set_xlabel('Year'); ax2.set_ylabel('AAR (%)')
fig.suptitle('Daily model: ELA and AAR - Aletschgletscher')
fig.tight_layout()
fig.savefig(PLOT_DIR / 'plot2_ela_aar.png', dpi=150)
plt.show()

In [ ]:
# ---- Plot 3: Daily vs monthly comparison ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.2, 4.3))
ax1.plot(yrs_d, mb_d[idx_al], '-o', ms=3, color='#4682B4', label='Daily')
ax1.plot(yrs_m, mb_m[idx_al_m], '--o', ms=3, color='#4682B4', label='Monthly')
ax1.plot([1991, 2020], [-1.216, -1.216], 'k:', label='Geodetic (Hugonnet et al. 2021)')
ax1.set_title('Aletschgletscher'); ax1.set_xlabel('Year'); ax1.set_ylabel('MB (m w.e.)'); ax1.set_ylim(-5, 3)
ax1.legend(fontsize=9)

ax2.plot(yrs_d, mb_d[idx_mo], '-o', ms=3, color='#D25A1E', label='Daily')
ax2.plot(yrs_m, mb_m[idx_mo_m], '--o', ms=3, color='#D25A1E', label='Monthly')
ax2.plot([1991, 2020], [-1.022, -1.022], 'k:', label='Geodetic (Hugonnet et al. 2021)')
ax2.set_title('Morteratschgletscher'); ax2.set_xlabel('Year'); ax2.set_ylabel('MB (m w.e.)'); ax2.set_ylim(-5, 3)
ax2.legend(fontsize=9)

fig.suptitle('Annual mass balance: daily vs monthly (1991-2020)')
fig.tight_layout()
fig.savefig(PLOT_DIR / 'plot3_daily_vs_monthly.png', dpi=150)
plt.show()

In [ ]:
# ---- Plot 4: Calibrated parameters ----
# Al-d=Aletsch daily, Al-m=Aletsch monthly, Mo-d=Morteratsch daily, Mo-m=Morteratsch monthly
bar_labels = ['Al-d', 'Al-m', 'Mo-d', 'Mo-m']
x = np.arange(4)
ddfs  = [cal_d['ddfs'][idx_al_c],  cal_m['ddfs'][idx_al_cm],  cal_d['ddfs'][idx_mo_c],  cal_m['ddfs'][idx_mo_cm]]
ddfi  = [cal_d['ddfi'][idx_al_c],  cal_m['ddfi'][idx_al_cm],  cal_d['ddfi'][idx_mo_c],  cal_m['ddfi'][idx_mo_cm]]
cprec = [cal_d['cprec'][idx_al_c], cal_m['cprec'][idx_al_cm], cal_d['cprec'][idx_mo_c], cal_m['cprec'][idx_mo_cm]]
colors = ['#4682B4', '#4682B4', '#D25A1E', '#D25A1E']

fig, axes = plt.subplots(1, 3, figsize=(10.2, 4.4))
for ax, vals, title, ylabel in zip(axes, [ddfs, ddfi, cprec],
                                    ['DDFsnow (mm/d/°C)', 'DDFice (mm/d/°C)', 'c_prec (-)'],
                                    ['mm/d/°C', 'mm/d/°C', '-']):
    ax.bar(x, vals, color=colors)
    ax.set_xticks(x); ax.set_xticklabels(bar_labels)
    ax.set_title(title); ax.set_ylabel(ylabel)
    ax.set_ylim(0, max(vals) * 1.2)

fig.suptitle('Calibrated parameters (Al=Aletsch, Mo=Morteratsch, d=daily, m=monthly)')
fig.text(0.5, 0.02, 'Aletsch', ha='right', color='#4682B4', fontweight='bold')
fig.text(0.51, 0.02, 'Morteratsch', ha='left', color='#D25A1E', fontweight='bold')
fig.tight_layout(rect=[0, 0.05, 1, 1])
fig.savefig(PLOT_DIR / 'plot4_calibrated_params.png', dpi=150)
plt.show()

## 5. Projection results (to 2100)

The projection runs (Steps 5 & 6) use GMIP4 / MRI-ESM2-0 / ssp126 with
`glacier_retreat='y'`, so glacier area is expected to decline substantially
over the 21st century.

In [ ]:
# ---- Read projected Area (daily and monthly) ----
_, yrs_proj_d, area_d = read_series(OUT_DIR / f'daily/CentralEurope/files/files_original/MRI-ESM2-0/ssp126/centraleurope_Area_r1{CC}.dat')
_, yrs_proj_m, area_m = read_series(OUT_DIR / f'monthly/CentralEurope/files/files_original/MRI-ESM2-0/ssp126/centraleurope_Area_r1{CC}.dat')

fig, ax = plt.subplots(figsize=(8.2, 4.2))
ax.plot(yrs_proj_d, area_d[idx_al], '-', color='#4682B4', label='Aletsch (daily)')
ax.plot(yrs_proj_d, area_d[idx_mo], '-', color='#D25A1E', label='Morteratsch (daily)')
ax.plot(yrs_proj_m, area_m[idx_al], '--', color='#4682B4', label='Aletsch (monthly)')
ax.plot(yrs_proj_m, area_m[idx_mo], '--', color='#D25A1E', label='Morteratsch (monthly)')
ax.set_xlabel('Year'); ax.set_ylabel('Area (km²)')
ax.set_title('Projected glacier area 1991-2100 (GMIP4 MRI-ESM2-0 ssp126)')
ax.legend(fontsize=9)
fig.tight_layout()
fig.savefig(PLOT_DIR / 'plot5_projected_area.png', dpi=150)
plt.show()

print(f"Aletsch area:     {area_d[idx_al, 0]:.1f} km2 (1991) -> {area_d[idx_al, -1]:.1f} km2 (2099), daily")
print(f"Morteratsch area: {area_d[idx_mo, 0]:.1f} km2 (1991) -> {area_d[idx_mo, -1]:.1f} km2 (2099), daily")

## 6. Sanity checks

Calibrated Ba within +/-0.3 m w.e./yr of the geodetic target, DDFs within
plausible physical bounds, and substantial projected area loss by 2100
indicate the model is working correctly.

In [ ]:
print()
print('====== Sanity checks ======')
n_pass = 0
n_total = 0

def check(label, value, lo, hi):
    global n_pass, n_total
    n_total += 1
    ok = lo <= value <= hi
    if ok:
        n_pass += 1
        print(f'  PASS  {label}: {value:7.3f}')
    else:
        print(f'  FAIL  {label}: {value:7.3f}  [expected {lo:.2f} to {hi:.2f}]')

check('Aletsch   daily  Ba',    cal_d['ba'][idx_al_c],  -1.5, -0.8)
check('Morteratsch daily  Ba',  cal_d['ba'][idx_mo_c],  -1.4, -0.5)
check('Aletsch   monthly Ba',   cal_m['ba'][idx_al_cm], -1.5, -0.8)
check('Morteratsch monthly Ba', cal_m['ba'][idx_mo_cm], -1.4, -0.5)

check('Aletsch daily DDFsnow', cal_d['ddfs'][idx_al_c], 1.5, 7.5)
check('Aletsch daily DDFice',  cal_d['ddfi'][idx_al_c], 3.0, 15.0)

ii_d = (yrs_d >= 2000) & (yrs_d <= 2020)
ii_m = (yrs_m >= 2000) & (yrs_m <= 2020)
check('Aletsch   daily  mean Ba 2000-2020',   mb_d[idx_al, ii_d].mean(),  -1.5, -0.8)
check('Morteratsch daily  mean Ba 2000-2020', mb_d[idx_mo, ii_d].mean(),  -1.4, -0.5)
check('Aletsch   monthly mean Ba 2000-2020',  mb_m[idx_al_m, ii_m].mean(), -1.5, -0.8)
check('Morteratsch monthly mean Ba 2000-2020',mb_m[idx_mo_m, ii_m].mean(), -1.4, -0.5)

check('Aletsch daily mean ELA 1991-2020', ela_d[idx_al].mean(), 2600, 3300)

# Projection retreat checks — area in 2100 should be well below 1991
check('Aletsch     2100 area loss (%, daily)',   100 * (1 - area_d[idx_al, -1] / area_d[idx_al, 0]),  15, 90)
check('Morteratsch 2100 area loss (%, daily)',   100 * (1 - area_d[idx_mo, -1] / area_d[idx_mo, 0]),  15, 90)

print()
print(f'{n_pass} / {n_total} checks passed')
if n_pass == n_total:
    print('All checks passed - the model is working correctly!')
else:
    print('Some checks failed - inspect the output files for details.')

## 7. Next steps

- **Your own region**: copy any test config to `config.pro`, change
  `catchment_selection`, `region_id_loop`, `dirres`, and the data paths
  to your production locations.
- **More GCMs / SSPs**: the projection configs use a single GMIP4
  model/scenario (MRI-ESM2-0 / ssp126) matching the bundled test data.
  For a full ensemble, set `MIP`, `GCM_model_idx`, `GCM_rcp_idx` per the
  [GCM configuration docs](../running-glogem/gcm-configuration.md) and
  point `dir_clim` at a full climate data share.
- **More physics**: uncomment `firnice_temperature='y'` or
  `debris_supraglacial='y'` in the config.